<a href="https://colab.research.google.com/github/ishanallasanagala/ds2002-fa26/blob/main/notebooks/06-midterm-walmart/2026-11-04%20%E2%80%94%20Reflection%20What%20Broke%20What%20Mattered%20%E2%80%94%20Studio.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Reflection What Broke What Mattered

**Studio — 2026-11-04 · Fall 2026**  
**Class time:** 45 minutes

---

## Reflection — what broke, what mattered

The project is submitted. This hour is for extracting what is reusable from it, because the capstone starts in under two weeks and it is the same work at larger scale.

Write individually for the first fifteen minutes, then compare with your team. The disagreements are the interesting part — if two people on the same team have different accounts of which decision mattered most, at least one of you learned something the other did not.

Answers here should be specific enough that someone could act on them. "Communicate better" is not usable. "Name one notebook owner per week, because we spent two hours reconciling two versions of the cleaning" is.

### 1. Data quality

Which single data-quality problem, if it had gone unnoticed, would have changed your conclusion the most? Give the number both ways.

_..._

### 2. ETL trade-offs

Name a cleaning decision where a reasonable person could have chosen the other way. What was the other option, what would it have done to your reported numbers, and would you make the same call again?

_..._

### 3. Pipeline trust

If someone handed you this notebook and asked whether the numbers are right, what would you check first? Is that check currently in the notebook? If not, that is your first capstone improvement.

_..._

### 4. Business versus data

Your recommendation went past what the data strictly proves — every useful recommendation does. Where exactly is that line in your work, and how would you say it out loud to someone who is about to spend money based on it?

_..._

### 5. Team process

What cost your team the most time that was not a data problem? Merge conflicts, duplicated work, waiting on someone, unclear ownership? Name the specific change you will make for the capstone.

_..._

### 6. Carry-forward list

**TODO:** three concrete things you are taking into the capstone. Code you will reuse, a habit you are adopting, or a mistake you are not repeating.

Run the cell — this list is your capstone kickoff agenda, so make it real.

In [5]:
carry_forward = [
    'TODO: something you will reuse (a function, a pattern, a template)',
    'TODO: a habit you are adopting',
    'TODO: a mistake you are not repeating',
]

for i, item in enumerate(carry_forward, 1):
    flag = 'unfilled' if item.startswith('TODO') else 'ok'
    print(f'{i}. [{flag}] {item}')
print()
print('bring this list to the capstone kickoff on Nov 16.')

1. [unfilled] TODO: something you will reuse (a function, a pattern, a template)
2. [unfilled] TODO: a habit you are adopting
3. [unfilled] TODO: a mistake you are not repeating

bring this list to the capstone kickoff on Nov 16.


---

## Checkpoint (participation)

Report the single data-quality problem that mattered most and the number it would have changed.

Work in groups if you wish, then fill in the cell below **yourself**. Paste the printed output (or a screenshot of it) into this week's **Studio Checkpoint** in Canvas by **Thursday 11:59pm ET**. One submission per person, not per group.

In [6]:
# Checkpoint
import os
import subprocess
from pathlib import Path
import pandas as pd

REPO_URL = "https://github.com/nishithy13/DS2002-Midterm.git"
REPO_PATH = "DS2002-Midterm"
if not os.path.isdir(REPO_PATH) and not Path("06-midterm-walmart/data").exists() and not Path("data").exists():
    subprocess.run(["git", "clone", REPO_URL, REPO_PATH], check=True)

def find_csv(name):
    for folder in (".", "data", "06-midterm-walmart/data", "DS2002-Midterm/06-midterm-walmart/data", "../data"):
        p = Path(folder) / f"{name}.csv"
        if p.exists():
            return p
    raise FileNotFoundError(f"Could not find {name}.csv")

# Clean transactions using our pipeline
raw = pd.read_csv(find_csv("walmart_transactions"))
df = raw.drop_duplicates().copy()
df = df.loc[~(df["quantity"] < 0)].copy()
df["unit_price"] = pd.to_numeric(df["unit_price"].astype("string").str.replace(r"[$,]", "", regex=True), errors="coerce")
df["timestamp"] = pd.to_datetime(df["timestamp"], format="mixed", errors="coerce")
df["store_id"] = (df["store_id"].astype("string").str.strip().str.upper()
                  .str.replace("_", "-", regex=False).str.replace(r"^FL(\d+)$", r"FL-\1", regex=True).replace("", pd.NA))
df = df.dropna(subset=["unit_price", "timestamp", "store_id"]).copy()
df["sku"] = df["sku"].astype("string").str.strip().str.upper()
/
# Compute Strawberry Pop-Tarts volume with SKU consolidation vs without (largest single SKU)
pt_skus = ["PT-12", "SKU#459812", "POPTART-STRAW", "PT_STRAWBERRY"]
pt_by_sku = df.loc[df["sku"].isin(pt_skus)].groupby("sku")["quantity"].sum()

problem_that_mattered = 'Fragmented SKU labels for Strawberry Pop-Tarts spread across 4 different codes (PT-12, POPTART-STRAW, PT_STRAWBERRY, SKU#459812)'
number_with_fix = int(pt_by_sku.sum())      # 5005 units across all 4 consolidated SKUs
number_without_fix = int(pt_by_sku.max())   # 1332 units for the single largest SKU (PT-12) alone
capstone_change = 'Lock the repo folder structure and multi-path find_csv() function at the start of the project, with one active notebook owner that pushes changes each day to minimize overlap errors'

print('problem:', problem_that_mattered)
print('reported:', number_with_fix, 'vs unfixed:', number_without_fix)
print('changing for the capstone:', capstone_change)

problem: Fragmented SKU labels for Strawberry Pop-Tarts spread across 4 different codes (PT-12, POPTART-STRAW, PT_STRAWBERRY, SKU#459812)
reported: 5005 vs unfixed: 1332
changing for the capstone: Lock the repo folder structure and multi-path find_csv() function at the start of the project, with one active notebook owner that pushes changes each day to minimize overlap errors
